# Solution 1 : passages complets

In [ ]:
import json
import math
import time
from pathlib import Path

import matplotlib.pyplot as plt
import gurobipy as gp
from gurobipy import GRB

DATASET = "data12"
NAME = "sol12_1"
TIME_LIMIT = 120


# chemins relatifs à src/ (dossier des notebooks)
DATA_DIR = Path("../data")
RESULTS_DIR = Path("../results")
DATA_FILE = DATA_DIR / f"{DATASET}.json"
COUNTRIES_FILE = DATA_DIR / "countries.geo.json"
REF_FILE = DATA_DIR / f"solution{DATASET[4:]}.json"  # solution fournie

with open(DATA_FILE) as f:
    data = json.load(f)

passes = data["Passes"]
accesses = data["Accesses"]
requests = data["ObservationRequests"]
D = data["minVisiDuration"]
FIXED = data["fixedPassCost"]
PER_UNIT = data["passCostPerTimeUnit"]

print(f"{DATA_FILE} : {len(passes)} passages, {len(accesses)} accès, {len(requests)} requêtes")

In [ ]:
def covers(start, end, acc):
    # recouvrement d'au moins D secondes
    return min(end, acc["end"]) - max(start, acc["start"]) >= D


def evaluate(data, solution):
    bookings = solution["Bookings"]
    ids = [b["passId"] for b in bookings]
    assert len(ids) == len(set(ids)), "deux réservations sur le même passage"

    cost, covered = 0.0, set()
    for b in bookings:
        p = passes[b["passId"]]
        assert p["start"] <= b["bookingStart"] <= b["bookingEnd"] <= p["end"], b
        cost += FIXED + PER_UNIT * (b["bookingEnd"] - b["bookingStart"])
        covered |= {a for a in p["accessIds"] if covers(b["bookingStart"], b["bookingEnd"], accesses[a])}

    sat, tot = {}, {}
    for r in requests:
        k = r["priority"]
        tot[k] = tot.get(k, 0) + 1
        sat[k] = sat.get(k, 0) + any(a in covered for a in r["accessIds"])
    aois = {accesses[a]["aoiId"] for a in covered}
    return {
        "cout": cost,
        "nb_passages": len(bookings),
        "requetes": f"{sum(sat.values())}/{len(requests)}",
        "par_priorite": {k: f"{sat[k]}/{tot[k]}" for k in sorted(tot)},
        "aois": f"{len(aois)}/{len(data['AOIs'])}",
    }


def solve_lexico(m, y, cost):
    # phase 1 : max requêtes satisfaites
    m.Params.TimeLimit = TIME_LIMIT
    m.setObjective(gp.quicksum(y.values()), GRB.MAXIMIZE)
    m.optimize()
    n_sat = round(m.ObjVal)
    t1 = m.Runtime
    print(f"phase 1 : {n_sat}/{len(requests)} requêtes ({t1:.2f}s)")

    # phase 2 : min coût sans perdre de requête
    m.addConstr(gp.quicksum(y.values()) >= n_sat, name="lexico")
    m.setObjective(cost, GRB.MINIMIZE)
    m.Params.TimeLimit = max(1, TIME_LIMIT - t1)
    m.optimize()
    print(f"phase 2 : coût {m.ObjVal:.0f}, borne {m.ObjBound:.0f}, gap {100 * m.MIPGap:.2f}% ({m.Runtime:.2f}s)")


def save_solution(bookings, name):
    path = RESULTS_DIR / f"{name}.json"
    with open(path, "w") as f:
        json.dump({"Bookings": bookings}, f)
    print("->", path)

In [ ]:
def plotSolution(data, solution):
    with open(COUNTRIES_FILE) as f:
        dataCountries = json.load(f)

    AOIs = data["AOIs"]
    satPasses = data["Passes"]
    bookings = sorted(solution["Bookings"], key=lambda b: satPasses[b["passId"]]["startDate"])
    for booking in bookings:
        print(booking)

    ncols = 3
    nrows = math.ceil(len(bookings) / ncols)
    fig, axs = plt.subplots(nrows=nrows, ncols=ncols, figsize=(30, 10 * nrows))
    axs = list(axs.flatten()) if hasattr(axs, "flatten") else [axs]

    for ax, booking in zip(axs, bookings):
        passId = booking["passId"]
        satPass = satPasses[passId]
        for feature in dataCountries["features"]:
            geometry = feature["geometry"]
            polygons = [geometry["coordinates"]] if geometry["type"] == "Polygon" else geometry["coordinates"]
            for polygon in polygons:
                for coords in polygon:
                    xs, ys = zip(*coords)
                    ax.fill(xs, ys, facecolor="lightgray", edgecolor="black", linewidth=0.5)
        ax.set_xlim(-9, 13)
        ax.set_ylim(38, 56)
        ax.set_title("Pass #{} [{},{}] réservé [{},{}]".format(
            passId, satPass["startDate"], satPass["endDate"], booking["bookingStart"], booking["bookingEnd"]))
        ax.plot([aoi["lon"] for aoi in AOIs], [aoi["lat"] for aoi in AOIs], "kx")

        xpoints, ypoints, colors = [], [], []
        for accessId in satPass["accessIds"]:
            access = accesses[accessId]
            aoi = AOIs[access["aoiId"]]
            xpoints.append(aoi["lon"])
            ypoints.append(aoi["lat"])
            # rouge = accès couvert par la réservation
            colors.append("r" if covers(booking["bookingStart"], booking["bookingEnd"], access) else "b")
        ax.scatter(xpoints, ypoints, marker="o", c=colors)

    for ax in axs[len(bookings):]:
        ax.set_visible(False)
    plt.show()

In [ ]:
m = gp.Model("passages_complets")

x = m.addVars(len(passes), vtype=GRB.BINARY, name="x")    # passage réservé
y = m.addVars(len(requests), vtype=GRB.BINARY, name="y")  # requête satisfaite

for r, req in enumerate(requests):
    P_r = {accesses[a]["passId"] for a in req["accessIds"]
           if accesses[a]["end"] - accesses[a]["start"] >= D}
    m.addConstr(y[r] <= gp.quicksum(x[p] for p in P_r), name=f"req{r}")

cout = gp.quicksum((FIXED + PER_UNIT * (p["end"] - p["start"])) * x[i] for i, p in enumerate(passes))
solve_lexico(m, y, cout)

In [ ]:
selected = [i for i in range(len(passes)) if x[i].X > 0.5]
print("passages :", selected)

# on réserve le passage entier
bookings = [{"passId": i, "passStart": passes[i]["start"], "passEnd": passes[i]["end"],
             "bookingStart": passes[i]["start"], "bookingEnd": passes[i]["end"]} for i in selected]
solution = {"Bookings": bookings}
save_solution(bookings, NAME)
evaluate(data, solution)

In [ ]:
plotSolution(data, solution)